<a href="https://colab.research.google.com/github/gschoie/ecos-fx-rates/blob/main/%EC%88%98%EC%B6%9C%EC%9E%85(%ED%99%95)_%EC%84%A0%EB%B0%95%EC%97%94%EC%A7%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 수출입 확정 — 선박엔진

관세청 품목별·국가별 수출입실적(공공데이터포털 `nitemtrade`) 2021년~현재.

| 소분류 | HS | 지역 묶음 |
|---|---|---|
| 주기 (선박 추진용 디젤, 이중연료 포함) | 8408.10 | 중국 / 일본 / 기타 |
| 주기 출력구간 | 8408.10-1000 / -2000 / -3000 | 중국 / 일본 / 기타 |
| 보기 (디젤 발전세트) | 8502.11 + 8502.12 + 8502.13 | 중국 / 일본 / 미국 / 기타 |

**실행 전 1회:** 왼쪽 🔑(보안 비밀)에 이름 `DATA_GO_KR_KEY`, 값에 공공데이터포털 인증키를 넣고 노트북 액세스를 켠다.
(저장소가 공개라 키를 코드에 적지 않는다.)

- '기타' 열에 묶인 개별 국가는 엑셀에서 접힌 열(+ 버튼)로 펼쳐 볼 수 있다.
- 8502(발전세트)에는 육상용 발전세트도 섞여 있다.
- 국내 조선소 탑재 엔진은 선박(8901 등)으로 수출돼 여기 잡히지 않는다.


In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import datetime
import time
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.formatting.rule import ColorScaleRule
import os

# ========================================================
# 1. 설정값 입력 (선박엔진 주기/보기 HS CODE 셋업)
# ========================================================
# 🔑 인증키는 코드에 적지 않는다 (공개 저장소).
#    Colab 왼쪽 🔑(보안 비밀)에 이름 DATA_GO_KR_KEY 로 등록 → 노트북 액세스 켜기
try:
    from google.colab import userdata
    API_KEY = userdata.get("DATA_GO_KR_KEY")
except Exception:
    API_KEY = os.environ.get("DATA_GO_KR_KEY", "")

# HS 코드 → (소분류, 묶을 지역). 같은 소분류끼리는 합산된다.
MAIN_REGIONS = ["중국", "일본"]            # 주기: 중국 / 일본 / 기타
AUX_REGIONS = ["중국", "일본", "미국"]     # 보기: 중국 / 일본 / 미국 / 기타

HS_CODES = {
    # 주기(Main Engine) — 선박 추진용 디젤(이중연료 포함)
    "840810": ("주기", MAIN_REGIONS),
    # 주기 출력 구간별 (HSK 10단위 — 조회 결과가 없으면 시트가 생략된다)
    "8408101000": ("주기_300kW이하", MAIN_REGIONS),
    "8408102000": ("주기_300~2000kW", MAIN_REGIONS),
    "8408103000": ("주기_2000kW초과", MAIN_REGIONS),
    # 보기(Aux Engine) — 디젤 발전세트 (육상용 발전세트도 섞여 있음)
    "850211": ("보기", AUX_REGIONS),
    "850212": ("보기", AUX_REGIONS),
    "850213": ("보기", AUX_REGIONS),
}
REGIONS_BY_SUB = {sub: regions for sub, regions in HS_CODES.values()}

current_year = datetime.today().year
years = list(range(2021, current_year + 1))

all_data = []

print("🚢 선박엔진 시계열 데이터(2021년~현재) 수집을 시작합니다...")
print(f"⚠️ 총 {len(HS_CODES)}개 HS 코드 수집이 진행됩니다.\n")
if not API_KEY:
    print("❌ 인증키가 없습니다. Colab 🔑 보안 비밀에 DATA_GO_KR_KEY 를 등록하세요.")

# ========================================================
# 2. 데이터 수집 (API 호출 파트)
# ========================================================
for hs_code, (item_name, _) in HS_CODES.items():
    print(f"🚀 [{item_name}] (HS: {hs_code}) 수집 중...")

    for year in years:
        start_month = f"{year}01"
        if year == current_year:
            end_month = datetime.today().strftime('%Y%m')
        else:
            end_month = f"{year}12"

        url = f"https://apis.data.go.kr/1220000/nitemtrade/getNitemtradeList?serviceKey={API_KEY}&strtYymm={start_month}&endYymm={end_month}&hsSgn={hs_code}&numOfRows=9999&pageNo=1"

        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                root = ET.fromstring(response.content)
                for item in root.findall(".//item"):
                    data = {
                        "대분류": "선박엔진",
                        "소분류": item_name,
                        "기간": item.findtext("year"),
                        "품목코드": item.findtext("hsCd"),
                        "품목명": item.findtext("statKor"),
                        "국가명": item.findtext("statCdCntnKor1"),
                        "수출금액": int(item.findtext("expDlr", "0")),
                        "수출중량": int(item.findtext("expWgt", "0")),
                        "수입금액(달러)": int(item.findtext("impDlr", "0")),
                        "수입중량(kg)": int(item.findtext("impWgt", "0")),
                        "무역수지(달러)": int(item.findtext("balPayments", "0"))
                    }
                    all_data.append(data)
            else:
                print(f"  ❌ {year}년 API 에러: {response.status_code}")
        except requests.exceptions.RequestException as e:
            print(f"  ⚠️ {year}년 네트워크 연결 실패: {e}")

        time.sleep(0.3)

# ========================================================
# 3. 데이터 가공 및 정제
# ========================================================
df = pd.DataFrame(all_data)

if not df.empty:
    print("\n🧹 데이터 정제 및 국가별 전세계 합계 생성 중...")
    df = df[df["기간"] != "총계"].copy()
    df = df[df["국가명"] != "총계"].copy()

    df["연"] = pd.to_datetime(df["기간"], format="%Y.%m", errors='coerce').dt.strftime("%Y년")
    df["YY.MM"] = pd.to_datetime(df["기간"], format="%Y.%m", errors='coerce')
    df = df.dropna(subset=['YY.MM'])

    # 분기 열 생성
    df['YY.QQ'] = df['YY.MM'].dt.to_period('Q').dt.strftime('%y.%qQ')

    # 월별용 인덱스
    min_date = df['YY.MM'].min()
    max_date = df['YY.MM'].max()
    full_date_range = pd.date_range(start=min_date, end=max_date, freq='MS')
    full_index = pd.MultiIndex.from_arrays([full_date_range.strftime("%Y년"), full_date_range], names=['연', 'YY.MM'])

    # 분기별용 인덱스
    min_q = df['YY.MM'].min().to_period('Q')
    max_q = df['YY.MM'].max().to_period('Q')
    full_q_range = pd.period_range(start=min_q, end=max_q, freq='Q')
    full_q_index = pd.MultiIndex.from_arrays([full_q_range.strftime("%Y년"), full_q_range.strftime("%y.%qQ")], names=['연', 'YY.QQ'])

    group_cols = ['대분류', '소분류', '연', 'YY.MM', 'YY.QQ', '기간']
    numeric_cols = ['수출금액', '수출중량', '수입금액(달러)', '수입중량(kg)', '무역수지(달러)']

    df_total = df.groupby(group_cols, dropna=False)[numeric_cols].sum().reset_index()
    df_total['국가명'] = '전세계(합계)'
    df_total['품목코드'] = '-'
    df_total['품목명'] = '소분류 통합'

    df = pd.concat([df_total, df], ignore_index=True)
    df['sort_key'] = df['국가명'].apply(lambda x: 0 if x == '전세계(합계)' else 1)
    df = df.sort_values(by=["소분류", "기간", "sort_key", "국가명"]).drop(columns=['sort_key']).reset_index(drop=True)

    # ========================================================
    # 4. 엑셀 파일 저장 (1, 2행 비우기 및 1행 높이 150 적용)
    # ========================================================
    file_name = "수출입통계_선박엔진_확정.xlsx"
    print(f"💾 엑셀 생성 중 (상단 이미지 삽입용 1~2행 확보 및 1행 높이 150 세팅)...")

    fill_amt = PatternFill(start_color="DCE6F1", end_color="DCE6F1", fill_type="solid")
    fill_wgt = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
    fill_asp = PatternFill(start_color="FCE4D6", end_color="FCE4D6", fill_type="solid")
    fill_yoy_amt = PatternFill(start_color="FFF2CC", end_color="FFF2CC", fill_type="solid")
    fill_gray = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")
    fill_total_row = PatternFill(start_color="EAEAEA", end_color="EAEAEA", fill_type="solid")

    color_scale_rule = ColorScaleRule(start_type='min', start_color='FFFFFF',
                                      end_type='max', end_color='FF9999')

    font_bold = Font(bold=True)
    align_center = Alignment(horizontal="center", vertical="center")
    align_left = Alignment(horizontal="left", vertical="center")

    with pd.ExcelWriter(file_name, engine='openpyxl') as writer:
        # 원본 로우 데이터 시트는 그대로 유지
        df.to_excel(writer, sheet_name='원천데이터(로우)', index=False)
        ws_raw = writer.sheets['원천데이터(로우)']

        # HS_CODES 에 적은 순서대로 (주기 → 출력구간 → 보기)
        unique_sub_categories = [s for s in REGIONS_BY_SUB if s in set(df['소분류'])]

        for sub_cat in unique_sub_categories:
            for time_type, time_col, time_full_index, shift_val in [
                ('월별', 'YY.MM', full_index, 12),
                ('분기별', 'YY.QQ', full_q_index, 4)
            ]:
                print(f"📊 피벗 서식 적용 중 ➡️ [{sub_cat} - {time_type}]")
                sub_df = df[df['소분류'] == sub_cat].copy()

                pivot_df = pd.pivot_table(
                    sub_df,
                    values=['수출금액', '수출중량'],
                    index=['연', time_col],
                    columns=['국가명'],
                    aggfunc='sum',
                    fill_value=0
                )

                pivot_df = pivot_df.reindex(time_full_index, fill_value=0)

                # 단위 변환
                pivot_df['수출금액'] = pivot_df['수출금액'] / 1000000
                pivot_df['수출중량'] = pivot_df['수출중량'] / 1000

                # 고정 지역 열 + '기타' 열 생성 (방산의 상위 5개국 대신 지정 지역으로 묶음)
                regions = REGIONS_BY_SUB[sub_cat]
                for region in regions:
                    for metric in ['수출금액', '수출중량']:
                        if (metric, region) not in pivot_df.columns:
                            pivot_df[(metric, region)] = 0.0

                countries = sub_df['국가명'].unique()
                country_totals = sub_df.groupby('국가명')['수출금액'].sum()
                remaining = [c for c in countries if c != '전세계(합계)' and c not in regions]
                remaining = sorted(remaining, key=lambda x: country_totals.get(x, 0), reverse=True)

                pivot_df[('수출금액', '기타')] = pivot_df['수출금액'][remaining].sum(axis=1) if remaining else 0.0
                pivot_df[('수출중량', '기타')] = pivot_df['수출중량'][remaining].sum(axis=1) if remaining else 0.0

                all_cols_countries = regions + ['기타'] + remaining + ['전세계(합계)']
                for country in all_cols_countries:
                    if ('수출금액', country) in pivot_df.columns:
                        amt = pivot_df[('수출금액', country)]
                        wgt = pivot_df[('수출중량', country)]
                        pivot_df[('ASP', country)] = (amt / wgt).fillna(0).replace([float('inf'), float('-inf')], 0).round(2)

                pivot_df_shifted = pivot_df.shift(shift_val)
                for country in all_cols_countries:
                    if ('수출금액', country) in pivot_df.columns:
                        amt_curr = pivot_df[('수출금액', country)]
                        amt_prev = pivot_df_shifted[('수출금액', country)]
                        pivot_df[('수출금액의 YoY', country)] = ((amt_curr - amt_prev) / amt_prev).fillna(0).replace([float('inf'), float('-inf')], 0)

                pivot_df = pivot_df.rename(columns={
                    '수출금액': '수출금액(백만달러)',
                    '수출중량': '수출중량(톤)',
                    'ASP': 'ASP(백만달러/톤)'
                })

                # 전세계 → 지정 지역 → 기타 → (기타에 묶인 개별국: 접힘)
                custom_country_order = ['전세계(합계)'] + regions + ['기타'] + remaining
                num_c = len(custom_country_order)

                sorted_cols = []
                metrics_order = ['수출금액(백만달러)', '수출중량(톤)', 'ASP(백만달러/톤)', '수출금액의 YoY']
                for metric in metrics_order:
                    for c in custom_country_order:
                        if (metric, c) in pivot_df.columns:
                            sorted_cols.append((metric, c))

                pivot_df = pivot_df.reindex(columns=sorted_cols)
                pivot_df.columns.names = [None, None]
                pivot_df.index.names = [None, None]

                sheet_name = f"피벗_{sub_cat}" if time_type == '월별' else f"피벗_분기_{sub_cat}"

                # 💡 [핵심 로직 1] startrow=2를 부여하여 데이터를 3행부터 강제 시작 (1, 2행 비움)
                pivot_df.to_excel(writer, sheet_name=sheet_name, startrow=2)
                ws_pivot = writer.sheets[sheet_name]

                # 💡 [핵심 로직 2] 1행의 높이를 150으로 조정하여 이미지 공간 확보
                ws_pivot.row_dimensions[1].height = 150

                # 레이아웃 세팅 (데이터가 2칸씩 밀렸으므로 좌표도 모두 +2 보정)
                ws_pivot.cell(row=4, column=1).value = "연"
                ws_pivot.cell(row=4, column=2).value = "YY.MM" if time_type == '월별' else "분기"

                # 💡 [핵심 로직 3] 데이터 시작점이 밀렸으므로 틀 고정도 C4에서 C6으로 변경
                ws_pivot.freeze_panes = 'C6'
                ws_pivot.sheet_properties.outlinePr.summaryRight = True

                # 💡 5행(기존 3행)에 합계 타이틀 기입
                ws_pivot.cell(row=5, column=1).value = "전체 누적"
                ws_pivot.cell(row=5, column=2).value = "합계"
                for col_idx in [1, 2]:
                    ws_pivot.cell(row=5, column=col_idx).font = font_bold
                    ws_pivot.cell(row=5, column=col_idx).fill = fill_total_row
                    ws_pivot.cell(row=5, column=col_idx).alignment = align_center

                # 데이터 영역 날짜 포맷 (6행부터)
                for row in range(6, ws_pivot.max_row + 1):
                    if time_type == '월별':
                        ws_pivot.cell(row=row, column=2).number_format = 'yy.mm'
                    else:
                        ws_pivot.cell(row=row, column=2).alignment = align_center

                # 3, 4행(기존 1, 2행) 헤더 서식
                for r in [3, 4]:
                    for c in [1, 2]:
                        cell = ws_pivot.cell(row=r, column=c)
                        cell.fill = fill_gray
                        cell.font = font_bold
                        cell.alignment = align_center

                # 컬럼 서식 루프 (수식과 조건부 서식 좌표를 모두 6행부터로 보정)
                for col in range(3, ws_pivot.max_column + 1):
                    metric_name = sorted_cols[col - 3][0]
                    country_name = sorted_cols[col - 3][1]
                    col_letter = get_column_letter(col)
                    last_row = ws_pivot.max_row

                    is_global_wgt_asp = (country_name == '전세계(합계)') and (metric_name in ['수출중량(톤)', 'ASP(백만달러/톤)'])

                    if remaining and country_name in remaining:
                        ws_pivot.column_dimensions[col_letter].outline_level = 1
                        ws_pivot.column_dimensions[col_letter].hidden = True

                    if metric_name == '수출금액(백만달러)':
                        current_fill, fmt = fill_amt, '#,##0.00'
                        # 💡 6행부터 합산
                        ws_pivot.cell(row=5, column=col).value = f"=SUM({col_letter}6:{col_letter}{last_row})"
                        ws_pivot.conditional_formatting.add(f"{col_letter}6:{col_letter}{last_row}", color_scale_rule)

                    elif metric_name == '수출중량(톤)':
                        current_fill, fmt = fill_wgt, '#,##0.00'
                        ws_pivot.cell(row=5, column=col).value = f"=SUM({col_letter}6:{col_letter}{last_row})"

                    elif metric_name == 'ASP(백만달러/톤)':
                        current_fill, fmt = fill_asp, '#,##0.00'
                        amt_col_letter = get_column_letter(col - (2 * num_c))
                        wgt_col_letter = get_column_letter(col - num_c)
                        # 💡 5행(합계행)을 참조하여 ASP 계산
                        ws_pivot.cell(row=5, column=col).value = f"=IFERROR({amt_col_letter}5/{wgt_col_letter}5, 0)"

                    elif metric_name == '수출금액의 YoY':
                        current_fill, fmt = fill_yoy_amt, '0.0%'
                        ws_pivot.cell(row=5, column=col).value = "-"
                        ws_pivot.conditional_formatting.add(f"{col_letter}6:{col_letter}{last_row}", color_scale_rule)

                    total_cell = ws_pivot.cell(row=5, column=col)
                    total_cell.fill = fill_total_row
                    total_cell.font = font_bold
                    total_cell.number_format = fmt

                    for r in [3, 4]:
                        cell = ws_pivot.cell(row=r, column=col)
                        cell.fill = current_fill
                        cell.font = font_bold
                        cell.alignment = align_left if r == 3 else align_center

                    # 6행부터 실제 데이터 포맷팅 적용
                    for row in range(6, last_row + 1):
                        cell = ws_pivot.cell(row=row, column=col)
                        cell.number_format = fmt
                        if is_global_wgt_asp:
                            cell.fill = fill_total_row
                            cell.font = font_bold

        # 원천데이터: 열 이름으로 찾아 서식 적용
        raw_cols = list(df.columns)
        yymm_col = raw_cols.index('YY.MM') + 1
        num_col_idx = [raw_cols.index(c) + 1 for c in numeric_cols]
        for row in range(2, ws_raw.max_row + 1):
            ws_raw.cell(row=row, column=yymm_col).number_format = 'yy.mm'
            for col_idx in num_col_idx:
                ws_raw.cell(row=row, column=col_idx).number_format = '#,##0'

        for sheet in writer.sheets.values():
            for row in sheet.iter_rows():
                for cell in row:
                    current_align = cell.alignment
                    if current_align:
                        cell.alignment = Alignment(
                            horizontal=current_align.horizontal,
                            vertical=current_align.vertical,
                            text_rotation=current_align.text_rotation,
                            wrap_text=current_align.wrap_text,
                            shrink_to_fit=True,
                            indent=current_align.indent
                        )
                    else:
                        cell.alignment = Alignment(shrink_to_fit=True)

    print(f"\n✅ 완료! 1행(높이 150)과 2행이 비워져 이미지 삽입 준비가 된 [ {file_name} ] 파일이 출력됩니다.")

    try:
        from google.colab import files
        files.download(file_name)
    except: pass
else:
    print("\n❌ 수집된 데이터가 없습니다.")
